In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings("ignore")
# Load data
df = pd.read_csv('data.csv')

if "Unnamed: 32" in df.columns:
    df = df.drop("Unnamed: 32", axis=1)
else:
    print("this is not available Unamed: 32")

df = df.drop('id', axis=1)
df['diagnosis'] = df['diagnosis'].map({'M': 1, 'B': 0})  # 1 = Malignant, 0 = Benign

X = df.drop('diagnosis', axis=1)
y = df['diagnosis']

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale (XGBoost scaling se farak nahi padta, but safe side ke liye kar do)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

: 

In [5]:
from sklearn.model_selection import GridSearchCV

xgb = XGBClassifier(eval_metric='logloss', use_label_encoder=False, random_state=42)

param_grid = {
    'n_estimators': [300, 500, 700],
    'max_depth': [3, 4, 5],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.8, 0.9, 1.0],
    'colsample_bytree': [0.8, 0.9, 1.0]
}

grid = GridSearchCV(xgb, param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)
grid.fit(X_train_scaled, y_train)

print("Best params:", grid.best_params_)
print("Best CV accuracy:", grid.best_score_)

# Test set evaluation
best_xgb = grid.best_estimator_
y_pred = best_xgb.predict(X_test_scaled)
test_acc = accuracy_score(y_test, y_pred)
print("Test accuracy:", test_acc)

Fitting 5 folds for each of 243 candidates, totalling 1215 fits
Best params: {'colsample_bytree': 0.9, 'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 500, 'subsample': 1.0}
Best CV accuracy: 0.9736263736263737
Test accuracy: 0.9649122807017544


In [6]:
if "Unnamed: 32" in df.columns:
    df = df.drop("Unnamed: 32", axis=1)
else:
    print("this is not available")

this is not available


In [7]:
if "id" in df.columns:
    df= df.drop ("id", axis=1)
df

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,1,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.30010,0.14710,0.2419,...,25.380,17.33,184.60,2019.0,0.16220,0.66560,0.7119,0.2654,0.4601,0.11890
1,1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.08690,0.07017,0.1812,...,24.990,23.41,158.80,1956.0,0.12380,0.18660,0.2416,0.1860,0.2750,0.08902
2,1,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.19740,0.12790,0.2069,...,23.570,25.53,152.50,1709.0,0.14440,0.42450,0.4504,0.2430,0.3613,0.08758
3,1,11.42,20.38,77.58,386.1,0.14250,0.28390,0.24140,0.10520,0.2597,...,14.910,26.50,98.87,567.7,0.20980,0.86630,0.6869,0.2575,0.6638,0.17300
4,1,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.19800,0.10430,0.1809,...,22.540,16.67,152.20,1575.0,0.13740,0.20500,0.4000,0.1625,0.2364,0.07678
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
564,1,21.56,22.39,142.00,1479.0,0.11100,0.11590,0.24390,0.13890,0.1726,...,25.450,26.40,166.10,2027.0,0.14100,0.21130,0.4107,0.2216,0.2060,0.07115
565,1,20.13,28.25,131.20,1261.0,0.09780,0.10340,0.14400,0.09791,0.1752,...,23.690,38.25,155.00,1731.0,0.11660,0.19220,0.3215,0.1628,0.2572,0.06637
566,1,16.60,28.08,108.30,858.1,0.08455,0.10230,0.09251,0.05302,0.1590,...,18.980,34.12,126.70,1124.0,0.11390,0.30940,0.3403,0.1418,0.2218,0.07820
567,1,20.60,29.33,140.10,1265.0,0.11780,0.27700,0.35140,0.15200,0.2397,...,25.740,39.42,184.60,1821.0,0.16500,0.86810,0.9387,0.2650,0.4087,0.12400


In [8]:
from sklearn.impute import SimpleImputer

# Numeric columns ke liye median imputer
imputer = SimpleImputer(strategy='median')
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

# Phir scaling karo
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

In [9]:
from sklearn.ensemble import VotingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

# Tuned models (apne best parameters use kar)
svm_best = SVC(C=10, gamma=0.01, kernel='rbf', probability=True, random_state=42)
xgb_best = XGBClassifier(**grid.best_params_, eval_metric='logloss', use_label_encoder=False, random_state=42)
lr_best = LogisticRegression(C=1, max_iter=1000)

voting = VotingClassifier(
    estimators=[('svm', svm_best), ('xgb', xgb_best), ('lr', lr_best)],
    voting='soft'
)

voting.fit(X_train_scaled, y_train)
y_pred_voting = voting.predict(X_test_scaled)
print("Voting Accuracy:", accuracy_score(y_test, y_pred_voting))

Voting AccuAracy: 0.9824561403508771


In [10]:
print("Missing values in X_train:", X_train.isnull().sum().sum())
print("Missing values in X_test:", X_test.isnull().sum().sum())

Missing values in X_train: 0
Missing values in X_test: 0


In [11]:
cv_scores = cross_val_score(best_xgb, X_train_scaled, y_train, cv=5)
print("CV scores:", cv_scores)
print("CV mean:", cv_scores.mean())

CV scores: [0.97802198 1.         0.93406593 1.         0.95604396]
CV mean: 0.9736263736263737


In [23]:
# Take first sample from test set
sample = X_test_scaled[0].reshape(1, -1)
pred = voting.predict(sample)
print("Predicted:", pred[0])
print("Actual:", y_test.iloc[0])

Predicted: 0
Actual: 0


In [24]:
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier

xgb = XGBClassifier(eval_metric='logloss', use_label_encoder=False, random_state=42)

param_grid = {
    'n_estimators': [500, 700, 1000],
    'max_depth': [3, 4],
    'learning_rate': [0.01, 0.02],
    'subsample': [0.9, 1.0],
    'colsample_bytree': [0.9, 1.0]
}

grid = GridSearchCV(xgb, param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)
grid.fit(X_train_scaled, y_train)

print("Best CV:", grid.best_score_)
print("Test accuracy:", accuracy_score(y_test, grid.predict(X_test_scaled)))

Fitting 5 folds for each of 48 candidates, totalling 240 fits
Best CV: 0.9692307692307693
Test accuracy: 0.9649122807017544


In [25]:
# List of all feature names (30 features) – exact order mein
feature_names = ['radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean',
                 'smoothness_mean', 'compactness_mean', 'concavity_mean',
                 'concave points_mean', 'symmetry_mean', 'fractal_dimension_mean',
                 'radius_se', 'texture_se', 'perimeter_se', 'area_se',
                 'smoothness_se', 'compactness_se', 'concavity_se',
                 'concave points_se', 'symmetry_se', 'fractal_dimension_se',
                 'radius_worst', 'texture_worst', 'perimeter_worst', 'area_worst',
                 'smoothness_worst', 'compactness_worst', 'concavity_worst',
                 'concave points_worst', 'symmetry_worst', 'fractal_dimension_worst']

# Function to take manual input and predict
def predict_manual(model, scaler, feature_names):
    print("\n🔍 Enter the values for the 30 features (one by one):\n")
    input_values = []
    for i, name in enumerate(feature_names):
        val = float(input(f"{i+1}. {name}: "))
        input_values.append(val)
    
    import numpy as np
    input_array = np.array(input_values).reshape(1, -1)
    input_scaled = scaler.transform(input_array)
    pred = model.predict(input_scaled)[0]
    proba = model.predict_proba(input_scaled)[0]
    confidence = proba[pred] * 100
    
    if pred == 1:
        print(f"\n⚠️ Prediction: **Malignant (Cancerous)** with {confidence:.2f}% confidence")
    else:
        print(f"\n✅ Prediction: **Benign (Non-cancerous)** with {confidence:.2f}% confidence")
    
    return pred

# Call this function (pehle apna model aur scaler load karo)
# predict_manual(voting, scaler, feature_names)

In [29]:
predict_manual(2.9, 9.0,8.2)


🔍 Enter the values for the 30 features (one by one):



TypeError: 'float' object is not iterable

In [30]:
# Pick any row from test set, change some values
sample = X_test.iloc[0].copy()  # original values (without scaling)
print("Original sample values:\n", sample)

# Modify values manually (example: radius_mean ko double kar do)
sample['radius_mean'] = sample['radius_mean'] * 2

# Scale karo aur predict karo
sample_scaled = scaler.transform([sample])
pred = voting.predict(sample_scaled)
print("Modified sample prediction:", "Malignant" if pred[0]==1 else "Benign")

Original sample values:
 radius_mean                 11.410000
texture_mean                10.820000
perimeter_mean              73.340000
area_mean                  403.300000
smoothness_mean              0.093730
compactness_mean             0.066850
concavity_mean               0.035120
concave points_mean          0.026230
symmetry_mean                0.166700
fractal_dimension_mean       0.061130
radius_se                    0.140800
texture_se                   0.460700
perimeter_se                 1.103000
area_se                     10.500000
smoothness_se                0.006040
compactness_se               0.015290
concavity_se                 0.015140
concave points_se            0.006460
symmetry_se                  0.013440
fractal_dimension_se         0.002206
radius_worst                12.820000
texture_worst               15.970000
perimeter_worst             83.740000
area_worst                 510.500000
smoothness_worst             0.154800
compactness_worst        

In [31]:
train_pred = voting.predict(X_train_scaled)
train_acc = accuracy_score(y_train, train_pred)
print("Training Accuracy:", train_acc)

Training Accuracy: 0.9934065934065934


In [32]:
import joblib
joblib.dump(voting, 'final_model.pkl')

['final_model.pkl']